# 07 — Embeddings: documento semántico por persona y búsqueda semántica

**Objetivo (DEC-014, ver `context/DECISION_LOG.md`):** construir, para cada persona, un
**documento semántico** — un texto narrativo coherente que integra su trayectoria (dentro y
fuera de ESPOL), formación, docencia, investigación, funciones adicionales/subrogaciones,
capacitación, idiomas y reconocimientos — y calcular UN embedding por persona a partir de
ESE documento (no promediando embeddings de fragmentos sueltos, como en la versión anterior
de este notebook).

Arquitectura:

```
datos → información integrada de la persona → DOCUMENTO SEMÁNTICO → embedding → índice vectorial → búsqueda semántica
```

Este notebook es independiente de `06_clustering.ipynb`: no reentrena ni modifica el
clustering existente, y sus salidas (`data/embeddings/`) no se concatenan con
`X_modelado.csv`. Las secciones 1-4 (inventario de fuentes de texto y corpus por fragmento)
se conservan como estaban — ese corpus por fragmento se sigue usando para mostrar evidencia
("por qué es relevante") en el dashboard, pero **ya no es el insumo del embedding**; el
embedding se calcula sobre el documento semántico construido en la sección 5.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd().parents[1] if (Path.cwd().name == "07_embeddings") else Path.cwd()
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_MODELING = ROOT / "data" / "modeling"
DATA_TRAYECTORIAS = ROOT / "data" / "trayectorias"
DATA_EMBEDDINGS = ROOT / "data" / "embeddings"
DATA_EMBEDDINGS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "notebooks" / "01_preprocesamiento"))
sys.path.insert(0, str(ROOT / "notebooks" / "07_embeddings"))
import _preprocesamiento_comun as pc
import _embeddings_comun as ec

RANDOM_STATE = 42

personas = pd.read_csv(DATA_MODELING / "personas_modelado.csv")
POBLACION = set(personas["IDPERSONA"])
print("Población de modelado (referencia para cobertura):", len(POBLACION))
print("Salidas de embeddings en:", DATA_EMBEDDINGS)

## 1. Columnas textuales disponibles en las fuentes institucionales

`data/features/*.csv` (usado en `03`-`05`) ya no tiene texto libre: todo fue agregado a conteos
numéricos por persona. El texto libre original vive en `data/processed/*.csv` (una fila por
publicación, capacitación, proyecto, etc., no por persona). Se revisan esas tablas en busca de
columnas de texto con contenido descriptivo real (no códigos, no IDs, no metadata administrativa).


In [ ]:
# Inventario de columnas candidatas identificadas por inspección de data/processed/*.csv
# (columnas de texto libre presentes en cada tabla, excluyendo IDs, fechas, referencias a archivos,
#  URLs, nombres/apellidos de personas y campos administrativos de auditoría/escalafón)
inventario_fuentes = pd.DataFrame([
    {"TABLA": "publicaciones.csv", "COLUMNA": "TITULO", "DESCRIPCION": "Título de publicación académica"},
    {"TABLA": "publicaciones.csv", "COLUMNA": "NOMBREREVISTA", "DESCRIPCION": "Nombre de la revista/venue (no describe el tema, sino el medio)"},
    {"TABLA": "proyecto_grado.csv", "COLUMNA": "NOMBRETRABAJOTITULACION", "DESCRIPCION": "Título del trabajo de titulación dirigido"},
    {"TABLA": "proyecto_grado.csv", "COLUMNA": "NOMBREPROGRAMA", "DESCRIPCION": "Programa académico del trabajo de titulación"},
    {"TABLA": "proyectos_investigacion_disponible.csv", "COLUMNA": "NOMBRE", "DESCRIPCION": "Nombre del proyecto de investigación"},
    {"TABLA": "proyectos_investigacion_disponible.csv", "COLUMNA": "STRAREACAMPOAMPLIO / STRAREAFRASCATI / STRSUBAREAFRASCATI", "DESCRIPCION": "Área de conocimiento del proyecto (texto descriptivo)"},
    {"TABLA": "proyectos_investigacion_disponible.csv", "COLUMNA": "STRCAMPOESPECIFICO", "DESCRIPCION": "Contiene códigos numéricos ('1.0','2.0'...), no texto real (problema de calidad de datos)"},
    {"TABLA": "proyectos_vinculacion_disponible.csv", "COLUMNA": "NOMBREPROYECTO / NOMBREPROGRAMA", "DESCRIPCION": "Nombre del proyecto/programa de vinculación con la comunidad"},
    {"TABLA": "ponentes_todos.csv", "COLUMNA": "NOMBRE", "DESCRIPCION": "Título de la ponencia/evento"},
    {"TABLA": "ponentes_todos.csv", "COLUMNA": "AREAACITACIONDESCRIPCION / TIPODESCRIPCION", "DESCRIPCION": "Códigos abreviados ('DI','PE','OT'), baja cobertura, no es texto descriptivo"},
    {"TABLA": "capacitaciones_todas.csv", "COLUMNA": "NOMBRE", "DESCRIPCION": "Nombre de la capacitación tomada"},
    {"TABLA": "capacitaciones_todas.csv", "COLUMNA": "AREACAPACITACIONDESCRIPCION", "DESCRIPCION": "Códigos abreviados, baja cobertura"},
    {"TABLA": "certificados_todos.csv", "COLUMNA": "NOMBRE", "DESCRIPCION": "Nombre de la certificación obtenida"},
    {"TABLA": "mencion_honor.csv", "COLUMNA": "NOMBREMENCION", "DESCRIPCION": "Categoría de la mención de honor (p.ej. 'Segunda mención'), no describe un tema"},
    {"TABLA": "experiencia_externa.csv", "COLUMNA": "CARGO", "DESCRIPCION": "Cargo ocupado en experiencia laboral externa"},
    {"TABLA": "experiencia_externa.csv", "COLUMNA": "INSTITUCION", "DESCRIPCION": "Nombre de la institución/empresa externa (identifica un organismo, no una competencia)"},
    {"TABLA": "carga_academica_disponible.csv", "COLUMNA": "NOMMATERIA", "DESCRIPCION": "Nombre de la materia impartida como docente"},
])
inventario_fuentes


## 2. Decisión: qué columnas se usan y por qué

Para cada tabla se mide, sobre la población de modelado (2213 personas), cuántas personas quedan
cubiertas por cada columna candidata, y se decide su inclusión con base en tres criterios: (a) si
describe realmente un **tema/competencia profesional o académica** (no un código, un medio o una
organización), (b) si tiene cobertura razonable, y (c) si no está prohibida por la consigna del
proyecto (IDs, nombres/apellidos, URLs, referencias a archivos, metadata administrativa).


In [ ]:
def cobertura(path, id_col, text_col, rename_id=None):
    df = pd.read_csv(DATA_PROCESSED / path, encoding="utf-8-sig")
    if rename_id:
        df = df.rename(columns={rename_id: id_col})
    df = df[df[id_col].isin(POBLACION) & df[text_col].notna()]
    return df[id_col].nunique(), len(df)

filas_decision = []

def registrar(fuente, tabla, columna, id_col, text_col, incluida, motivo, rename_id=None):
    n_personas, n_registros = cobertura(tabla, id_col, text_col, rename_id=rename_id)
    filas_decision.append({
        "FUENTE": fuente, "TABLA": tabla, "COLUMNA": text_col,
        "N_REGISTROS": n_registros, "N_PERSONAS_COBERTURA": n_personas,
        "PCT_POBLACION": round(100 * n_personas / len(POBLACION), 1),
        "INCLUIDA": incluida, "MOTIVO": motivo,
    })

registrar("PUBLICACION", "publicaciones.csv", "TITULO", "IDPERSONA", "TITULO",
          True, "Título describe directamente el tema de investigación")
registrar("PROYECTO_GRADO_DIRIGIDO", "proyecto_grado.csv", "NOMBRETRABAJOTITULACION", "IDPERSONA", "NOMBRETRABAJOTITULACION",
          True, "Título de tesis dirigida describe área de especialidad", rename_id="IDDIRECTOR")
registrar("PROYECTO_INVESTIGACION", "proyectos_investigacion_disponible.csv", "NOMBRE", "IDPERSONA", "NOMBRE",
          True, "Nombre de proyecto describe el tema investigado")
registrar("PROYECTO_VINCULACION", "proyectos_vinculacion_disponible.csv", "NOMBREPROYECTO", "IDPERSONA", "NOMBREPROYECTO",
          True, "Nombre de proyecto de vinculación describe el tema/comunidad de trabajo")
registrar("PONENCIA", "ponentes_todos.csv", "NOMBRE", "IDPERSONA", "NOMBRE",
          True, "Título de ponencia describe el tema presentado")
registrar("CAPACITACION", "capacitaciones_todas.csv", "NOMBRE", "IDPERSONA", "NOMBRE",
          True, "Nombre de capacitación describe el área de formación continua; cobertura muy alta")
registrar("CERTIFICACION", "certificados_todos.csv", "NOMBRE", "IDPERSONA", "NOMBRE",
          True, "Nombre de certificación describe una competencia adquirida")
registrar("EXPERIENCIA_EXTERNA_CARGO", "experiencia_externa.csv", "CARGO", "IDPERSONA", "CARGO",
          True, "Cargo externo describe rol/dominio profesional fuera de ESPOL")
registrar("MATERIA_IMPARTIDA", "carga_academica_disponible.csv", "NOMMATERIA", "IDPERSONA", "NOMMATERIA",
          True, "Nombre de materia impartida describe el área de docencia")
registrar("MENCION_HONOR", "mencion_honor.csv", "NOMBREMENCION", "IDPERSONA", "NOMBREMENCION",
          False, "Es una categoría de reconocimiento ('Diploma de honor'), no un tema/competencia")

decision_fuentes = pd.DataFrame(filas_decision).sort_values("PCT_POBLACION", ascending=False)
decision_fuentes


**Columnas descartadas explícitamente** (no llegan a la tabla de decisión porque no califican
como texto descriptivo, o violan las restricciones del alcance):

- `NOMBREREVISTA` (publicaciones) y `INSTITUCION` (experiencia externa): nombran un medio/organismo,
  no una competencia o tema — quedarían más cerca de "metadata administrativa" que de contenido
  profesional/académico.
- `STRCAMPOESPECIFICO` (proyectos de investigación): pese al nombre de la columna, contiene códigos
  numéricos como texto ('1.0', '2.0'), no descripciones — problema de calidad de datos, no una
  fuente de texto real.
- `AREAACITACIONDESCRIPCION`, `TIPODESCRIPCION` (ponentes) y `AREACAPACITACIONDESCRIPCION`
  (capacitaciones): son códigos abreviados ('DI', 'PE', 'OT') con cobertura muy baja, no texto
  descriptivo utilizable.
- Todo identificador (`IDPERSONA`, `IDCAPACITACION`, etc.), nombre/apellido de persona
  (`NOMBRES`, `APELLIDOS` en `heteroevaluacion_disponible.csv`), URL (`URLPUBLICACION`), referencia
  a archivo (`REFARCHIVO*`, `NAMEARCHDOC`) y campo administrativo de auditoría/escalafón
  (`ENESCALAFON`, `REVISADOPARAESCALAFON`, `IDUSUARIO`, `INGRESORRHH`, `ORIGENINGRESO`,
  `FECHASUBIDAARCHIVO`, etc.) — excluidos por instrucción explícita del proyecto.
- No se encontraron campos de comentarios/evaluaciones cualitativas de CENACAD en las tablas
  disponibles en `data/processed/` (solo el promedio numérico de heteroevaluación, ya incorporado
  en `X_modelado` como `PROMEDIO_HETEROEVALUACION`); si existen en otra fuente institucional no
  integrada aún, deben añadirse como una fuente adicional en una futura iteración.


## 3. Construcción del corpus por fragmento (evidencia/UI, no el embedding)

Se construye una tabla de detalle (una fila por *registro* de texto: cada capacitación,
publicación, proyecto, etc. es una fila separada). **Desde DEC-014, esta tabla ya no
alimenta el embedding** (ver sección 5) — se conserva porque el dashboard la usa para
mostrar fragmentos de evidencia ("por qué esta persona es relevante para tu búsqueda") sin
tener que re-parsear el documento semántico completo.

In [ ]:
def cargar(path):
    return pd.read_csv(DATA_PROCESSED / path, encoding="utf-8-sig")

registros = []

def agregar(df, id_col, fuente, texto_fn, rename_id=None):
    if rename_id:
        df = df.rename(columns={rename_id: id_col})
    df = df[df[id_col].isin(POBLACION)]
    for _, r in df.iterrows():
        texto = texto_fn(r)
        if texto and pd.notna(texto) and str(texto).strip():
            registros.append((int(r[id_col]), fuente, str(texto).strip()))

agregar(cargar("publicaciones.csv"), "IDPERSONA", "PUBLICACION",
        lambda r: r["TITULO"] if pd.notna(r["TITULO"]) else None)

agregar(cargar("proyecto_grado.csv"), "IDPERSONA", "PROYECTO_GRADO_DIRIGIDO",
        lambda r: (r["NOMBRETRABAJOTITULACION"] + (f" ({r['NOMBREPROGRAMA']})" if pd.notna(r.get("NOMBREPROGRAMA")) else ""))
                  if pd.notna(r["NOMBRETRABAJOTITULACION"]) else None,
        rename_id="IDDIRECTOR")

agregar(cargar("proyectos_investigacion_disponible.csv"), "IDPERSONA", "PROYECTO_INVESTIGACION",
        lambda r: " — ".join([str(r["NOMBRE"])] + [str(r[c]) for c in
                  ["STRAREACAMPOAMPLIO", "STRAREAFRASCATI", "STRSUBAREAFRASCATI"] if pd.notna(r.get(c))])
                  if pd.notna(r["NOMBRE"]) else None)

agregar(cargar("proyectos_vinculacion_disponible.csv"), "IDPERSONA", "PROYECTO_VINCULACION",
        lambda r: r["NOMBREPROYECTO"] + (f" — {r['NOMBREPROGRAMA']}" if pd.notna(r.get("NOMBREPROGRAMA")) else "")
                  if pd.notna(r["NOMBREPROYECTO"]) else None)

agregar(cargar("ponentes_todos.csv"), "IDPERSONA", "PONENCIA",
        lambda r: r["NOMBRE"] if pd.notna(r["NOMBRE"]) else None)

agregar(cargar("capacitaciones_todas.csv"), "IDPERSONA", "CAPACITACION",
        lambda r: r["NOMBRE"] if pd.notna(r["NOMBRE"]) else None)

agregar(cargar("certificados_todos.csv"), "IDPERSONA", "CERTIFICACION",
        lambda r: r["NOMBRE"] if pd.notna(r["NOMBRE"]) else None)

agregar(cargar("experiencia_externa.csv"), "IDPERSONA", "EXPERIENCIA_EXTERNA_CARGO",
        lambda r: r["CARGO"] if pd.notna(r["CARGO"]) else None)

df_materias = cargar("carga_academica_disponible.csv")
df_materias = df_materias[df_materias["IDPERSONA"].isin(POBLACION) & df_materias["NOMMATERIA"].notna()]
df_materias = df_materias.drop_duplicates(subset=["IDPERSONA", "NOMMATERIA"])
agregar(df_materias, "IDPERSONA", "MATERIA_IMPARTIDA", lambda r: r["NOMMATERIA"])

corpus_detalle = pd.DataFrame(registros, columns=["IDPERSONA", "FUENTE", "TEXTO"])
corpus_detalle = corpus_detalle.drop_duplicates(subset=["IDPERSONA", "FUENTE", "TEXTO"]).reset_index(drop=True)

print("Registros de texto totales:", len(corpus_detalle))
print("Personas con al menos un registro de texto:", corpus_detalle["IDPERSONA"].nunique(),
      f"de {len(POBLACION)} ({100*corpus_detalle['IDPERSONA'].nunique()/len(POBLACION):.1f}%)")
corpus_detalle["FUENTE"].value_counts()


In [ ]:
corpus_detalle.to_csv(DATA_EMBEDDINGS / "corpus_texto_detalle.csv", index=False)
print("Guardado:", DATA_EMBEDDINGS / "corpus_texto_detalle.csv", "-", corpus_detalle.shape)


## 4. Diagnóstico de cobertura del corpus

Se resume, por persona, cuántos registros de texto tiene y de cuántas fuentes distintas, **sin**
guardar el texto en este resumen (el texto vive únicamente en `corpus_texto_detalle.csv`, que es
insumo de trabajo, no un producto final).


In [ ]:
resumen_personas = (
    corpus_detalle.groupby("IDPERSONA")
    .agg(N_REGISTROS_TEXTO=("TEXTO", "size"), N_FUENTES_DISTINTAS=("FUENTE", "nunique"))
    .reset_index()
)
resumen_personas["LONGITUD_TOTAL_CARACTERES"] = (
    corpus_detalle.groupby("IDPERSONA")["TEXTO"].apply(lambda s: s.str.len().sum()).values
)

cobertura_completa = personas[["IDPERSONA"]].merge(resumen_personas, on="IDPERSONA", how="left")
cobertura_completa[["N_REGISTROS_TEXTO", "N_FUENTES_DISTINTAS", "LONGITUD_TOTAL_CARACTERES"]] = (
    cobertura_completa[["N_REGISTROS_TEXTO", "N_FUENTES_DISTINTAS", "LONGITUD_TOTAL_CARACTERES"]].fillna(0)
)
cobertura_completa.to_csv(DATA_EMBEDDINGS / "cobertura_texto_personas.csv", index=False)

sin_texto = (cobertura_completa["N_REGISTROS_TEXTO"] == 0).sum()
print(f"Personas sin ningún registro de texto: {sin_texto} ({100*sin_texto/len(cobertura_completa):.1f}%)")
cobertura_completa[["N_REGISTROS_TEXTO", "N_FUENTES_DISTINTAS", "LONGITUD_TOTAL_CARACTERES"]].describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(cobertura_completa["N_REGISTROS_TEXTO"].clip(upper=100), bins=40, color="#4C72B0")
axes[0].set_title("Registros de texto por persona (recortado en 100)")
axes[0].set_xlabel("N° de registros de texto")

fuente_counts = corpus_detalle["FUENTE"].value_counts()
axes[1].barh(fuente_counts.index[::-1], fuente_counts.values[::-1], color="#55A868")
axes[1].set_title("Registros de texto por fuente")
axes[1].set_xlabel("N° de registros")

plt.tight_layout()
plt.show()


**Lectura del diagnóstico:** el corpus cubre ~99% de la población de modelado con al menos un
registro de texto, dominado en volumen por `CAPACITACION` (más de la mitad de los registros) seguido
de `PUBLICACION`, `PROYECTO_INVESTIGACION`, `EXPERIENCIA_EXTERNA_CARGO` y `PROYECTO_GRADO_DIRIGIDO`.
Esto confirma que **sí existe suficiente texto institucional relevante** como para justificar
explorar una representación semántica — pero también advierte que, al calcular los embeddings, no
conviene concatenar todo el texto de una persona en un solo string (algunas personas superan 10,000
palabras): la mayoría de los modelos de embeddings truncan a unos pocos cientos de tokens. El diseño
más apropiado es **calcular un embedding por registro de texto y luego agregarlo (p.ej. promedio)
por persona**, posiblemente ponderando por fuente para que `CAPACITACION` no domine desproporcionadamente
solo por su volumen. Esta decisión de diseño se aplicará en la siguiente iteración, junto con el
modelo de embeddings elegido.


## 5. Documento semántico por persona (DEC-014)

Antes de calcular ningún embedding, se construye el **documento semántico**: un texto por
persona, ensamblado por secciones con información real (no se fuerzan secciones vacías),
que integra:

- **Trayectoria** — línea de tiempo cronológica combinando cargo estructural dentro de
  ESPOL (`tramos_rol.csv`, DEC-011), funciones adicionales/subrogaciones (`funciones_
  adicionales_persona.csv`, DEC-012, excluyendo las que coinciden con el contrato vigente)
  y experiencia laboral externa (`experiencia_externa.csv`) — en ese orden temporal real
  cuando las fechas lo permiten; no se inventan secuencias si faltan fechas.
- **Formación académica** (solo titulaciones con `Estado='Graduado'`).
- **Docencia** (materias impartidas, deduplicadas, con rango de años).
- **Investigación** (proyectos, publicaciones, tesis dirigidas, ponencias).
- **Vinculación** (proyectos de vinculación con la sociedad).
- **Capacitación** (cursos y certificaciones).
- **Idiomas** (excluyendo la lengua nativa).
- **Reconocimientos** (menciones de honor).

Implementado en `_embeddings_comun.py` (`construir_documentos_semanticos`), **separado**
de `_preprocesamiento_comun.py` para no mezclar esta lógica con las features del
clustering. Reutiliza `pc.construir_eventos_trayectoria` (ya calculada en
`04_trayectorias.ipynb`) en vez de reconstruir la línea de tiempo desde cero.

**Qué NO entra al documento (y por qué):**

| Se descarta de | Ejemplos | Motivo |
|---|---|---|
| El texto (queda solo en la fuente original) | IDs, `RMU`/salario, URLs, referencias a archivo, campos de auditoría (`ENESCALAFON`, `IDUSUARIO`...), nombres/apellidos | Sin valor semántico para búsqueda por tema/experiencia, o dato sensible/administrativo (ver DEC-003 sobre privacidad) |
| El texto (queda como metadata estructurada aparte) | `IDPERSONA`, `SECCIONES_INCLUIDAS`, `N_PALABRAS` | Útiles para filtrar/depurar resultados de búsqueda, pero no aportan significado semántico en sí mismos |
| El texto (se narra distinto) | Menciones de honor con categoría genérica sin tema | Se narran como reconocimiento con fecha/institución, no como "tema" de búsqueda |

In [ ]:
eventos_trayectoria = pd.read_csv(DATA_TRAYECTORIAS / "eventos_trayectoria_persona.csv")
# pd.to_datetime explicito, no parse_dates de read_csv (ver nota de robustez en
# construir_eventos_trayectoria / _preprocesamiento_comun.py, misma leccion de esta sesion).
for _c in ["FECHA_INICIO", "FECHA_FIN"]:
    eventos_trayectoria[_c] = pd.to_datetime(eventos_trayectoria[_c], format="mixed", errors="coerce")

# Feature engineering de movilidad de carrera (ver DECISION_LOG.md): ENTROPIA_CATEGORIA_CARGO
# habilita una oracion condicional de "alta diversidad de roles" solo para el tercil
# superior de la poblacion (ver _seccion_trayectoria) - se pasa aparte de eventos_trayectoria
# porque vive en features_trayectoria_persona.csv (04_trayectorias.ipynb), no en la tabla de
# eventos.
diversidad_trayectoria = pd.read_csv(
    DATA_TRAYECTORIAS / "features_trayectoria_persona.csv",
    usecols=["IDPERSONA", "ENTROPIA_CATEGORIA_CARGO", "N_CATEGORIAS_ROL_DISTINTAS",
             "TURBULENCIA_TRAMOS", "DURACION_MEDIANA_TRAMO_ANIOS"],
)

documentos = ec.construir_documentos_semanticos(POBLACION, eventos_trayectoria, diversidad_trayectoria)

documentos.to_csv(DATA_EMBEDDINGS / "documento_semantico_persona.csv", index=False, encoding="utf-8-sig")

vacios = (documentos["N_SECCIONES"] == 0).sum()
recortados = (documentos["SECCIONES_RECORTADAS"] != "").sum()
print(f"Guardado: {DATA_EMBEDDINGS / 'documento_semantico_persona.csv'} — {documentos.shape}")
print(f"Personas sin ninguna sección con información (documento vacío): {vacios} ({100*vacios/len(documentos):.1f}%)")
print(f"Personas con alguna sección recortada por presupuesto de longitud: {recortados} ({100*recortados/len(documentos):.1f}%)")
print()
print("Distribución de secciones incluidas por persona:")
print(documentos["N_SECCIONES"].value_counts().sort_index())
print()
print("Palabras por documento:")
print(documentos["N_PALABRAS"].describe())
print()
print("--- Ejemplo de documento (persona con más secciones) ---")
ejemplo = documentos.loc[documentos["N_SECCIONES"].idxmax()]
print(f"IDPERSONA {ejemplo['IDPERSONA']} ({ejemplo['N_PALABRAS']} palabras, secciones: {ejemplo['SECCIONES_INCLUIDAS']})")
print(ejemplo["DOCUMENTO_TEXTO"])

## 6. Cálculo del embedding (un vector por persona, a partir del documento semántico)

**Cambio de modelo respecto a la versión anterior de este notebook (decisión técnica,
DEC-014):** el modelo previo, `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`,
tiene un límite de contexto de **128 tokens** — adecuado para los fragmentos cortos que se
promediaban antes, pero insuficiente para un documento narrativo completo (mediana ~170
palabras, algunos documentos superan 300-400 palabras incluso después de recortar por
presupuesto en la sección 5): el texto se truncaría casi de inmediato, perdiendo la mayor
parte del contenido. Se cambia a **`intfloat/multilingual-e5-base`** (768 dimensiones,
contexto de **512 tokens**, multilingüe, entrenado específicamente para tareas de
recuperación/búsqueda semántica — no solo similitud de parafraseo). Sigue siendo un modelo
local (`sentence-transformers`), sin API key, sin enviar datos a terceros — se mantiene el
principio de privacidad de DEC-002. Los modelos E5 requieren un prefijo distinto para
documentos ("passage: ") y para consultas de búsqueda ("query: "); se aplica de forma
consistente aquí y en `notebooks/08_dashboard/app.py`.

A diferencia de la versión anterior, **no hay agregación**: cada persona tiene un único
texto (su documento semántico) y por lo tanto un único embedding directo — no se promedian
fragmentos ni se pondera por fuente.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-base"
model = SentenceTransformer(MODEL_NAME)

con_texto = documentos[documentos["DOCUMENTO_TEXTO"].str.len() > 0].copy()
sin_embedding = sorted(POBLACION - set(con_texto["IDPERSONA"]))

textos_passage = ("passage: " + con_texto["DOCUMENTO_TEXTO"]).tolist()
print(f"Codificando {len(textos_passage)} documentos (uno por persona)...")

embeddings_matrix = model.encode(
    textos_passage, batch_size=32, show_progress_bar=True, normalize_embeddings=True,
)
DIM_EMBEDDING = embeddings_matrix.shape[1]
ids_con_texto = con_texto["IDPERSONA"].to_numpy()

print("Embeddings calculados:", embeddings_matrix.shape)
print(f"Personas SIN embedding (documento vacío, ninguna sección con información): {len(sin_embedding)}")

In [ ]:
embeddings_personas = pd.DataFrame(
    embeddings_matrix, columns=[f"E_{i:03d}" for i in range(DIM_EMBEDDING)],
)
embeddings_personas.insert(0, "IDPERSONA", ids_con_texto)

embeddings_personas.to_csv(DATA_EMBEDDINGS / "embeddings_personas.csv", index=False)

print(f"Guardado: {DATA_EMBEDDINGS / 'embeddings_personas.csv'} — {embeddings_personas.shape}")
print(f"Personas SIN embedding (documento vacío): {len(sin_embedding)}")
print("IDs:", sin_embedding[:10], "..." if len(sin_embedding) > 10 else "")

In [ ]:
metadata_embeddings = pd.DataFrame([{
    "MODELO": MODEL_NAME,
    "DIMENSIONES": DIM_EMBEDDING,
    "METODO": "un embedding por persona, calculado directamente sobre su documento semántico "
              "completo (ver sección 5) - sin agregación de fragmentos (DEC-014)",
    "PREFIJO_DOCUMENTO": "passage: ",
    "PREFIJO_CONSULTA_BUSQUEDA": "query: ",
    "N_PERSONAS_CON_EMBEDDING": len(ids_con_texto),
    "N_PERSONAS_SIN_EMBEDDING": len(sin_embedding),
    "PALABRAS_PROMEDIO_DOCUMENTO": round(documentos["N_PALABRAS"].mean(), 1),
}])
metadata_embeddings.to_csv(DATA_EMBEDDINGS / "embeddings_metadata.csv", index=False)
metadata_embeddings.T

### Validación cualitativa: vecinos más cercanos en el espacio semántico

Antes de cualquier análisis cuantitativo, una revisión cualitativa rápida: para una persona con
suficiente texto (varias publicaciones/proyectos), ¿sus vecinos más cercanos por similitud coseno
tienen un perfil temático parecido? Esto no es una prueba formal, es una verificación de sentido
común de que el embedding capturó algo razonable.


In [ ]:
doc_por_persona = documentos.set_index("IDPERSONA")["DOCUMENTO_TEXTO"]
candidatos = documentos[documentos["N_SECCIONES"] >= 5]["IDPERSONA"]
persona_ejemplo = int(pd.Series(sorted(set(candidatos) & set(ids_con_texto))).sample(1, random_state=7).iloc[0])

idx_ejemplo = list(ids_con_texto).index(persona_ejemplo)
sims = embeddings_matrix @ embeddings_matrix[idx_ejemplo]
orden = np.argsort(-sims)
vecinos = [ids_con_texto[i] for i in orden[1:6]]

def resumen_texto(idp, max_chars=220):
    texto = doc_por_persona.get(idp, "")
    return texto.replace(chr(10), " | ")[:max_chars]

print(f"Persona de referencia {persona_ejemplo}:")
print(" ", resumen_texto(persona_ejemplo))
print()
print("Vecinos mas cercanos (similitud coseno, documento semantico completo):")
for v in vecinos:
    print(f"  [{sims[list(ids_con_texto).index(v)]:.3f}] {v}: {resumen_texto(v)}")

## 8. ¿Los embeddings identifican una estructura distinta a la de `06_clustering`?

Se agrupa el espacio de embeddings por sí solo (K-Means exploratorio, K=5) y se compara,
vía ARI, contra la jerarquía estructural actual (`data/clustering/clusters_personas.csv`,
columna `CLUSTER` — 13 categorías de cargo real, DEC-009, ya no un K-Means; ARI no requiere
que ambas particiones tengan el mismo número de grupos). La comparación se limita a las
personas que tienen documento semántico con al menos una sección (no todas las 3195 tienen
embedding).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA

clusters_estructurales = pd.read_csv(ROOT / "data" / "clustering" / "clusters_personas.csv")

X_emb = embeddings_matrix  # ya normalizado por fila

filas_emb_k = []
for k in range(3, 9):
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_emb)
    filas_emb_k.append({"K": k, "SILHOUETTE": silhouette_score(X_emb, labels)})
pd.DataFrame(filas_emb_k)

Se usa **K=5** para el agrupamiento exploratorio del espacio de embeddings — un valor
razonable para inspeccionar la estructura semántica por sí sola, sin intentar igualar el
número de categorías de la jerarquía estructural actual (13 categorías de cargo real,
DEC-009): ARI compara la concordancia entre dos particiones sin importar que tengan
cardinalidades distintas.

In [ ]:
labels_emb = KMeans(n_clusters=5, n_init=10, random_state=RANDOM_STATE).fit_predict(X_emb)

emb_clusters_df = pd.DataFrame({"IDPERSONA": ids_con_texto, "CLUSTER_EMBEDDING": labels_emb})
comparacion = emb_clusters_df.merge(clusters_estructurales, on="IDPERSONA", how="inner")
comparacion = comparacion.rename(columns={"CLUSTER": "CLUSTER_ESTRUCTURAL"})

ari = adjusted_rand_score(comparacion["CLUSTER_ESTRUCTURAL"], comparacion["CLUSTER_EMBEDDING"])
print(f"Personas comparadas: {len(comparacion)}")
print(f"ARI (estructural [13 categorías, DEC-009] vs. embeddings [K=5 exploratorio]): {ari:.3f}")
print()
tabla_cruzada = pd.crosstab(comparacion["CLUSTER_ESTRUCTURAL"], comparacion["CLUSTER_EMBEDDING"])
tabla_cruzada


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

pca_emb = PCA(n_components=2, random_state=RANDOM_STATE)
X_emb_pca = pca_emb.fit_transform(X_emb)

paleta5 = sns.color_palette("Set2", 5)
merged_pca = pd.DataFrame(X_emb_pca, columns=["PC1", "PC2"])
merged_pca["IDPERSONA"] = ids_con_texto
merged_pca = merged_pca.merge(comparacion, on="IDPERSONA", how="left")

for c in range(5):
    m = merged_pca["CLUSTER_EMBEDDING"] == c
    axes[0].scatter(merged_pca.loc[m, "PC1"], merged_pca.loc[m, "PC2"], s=10, alpha=0.6, color=paleta5[c], label=f"C{c}")
axes[0].set_title("Espacio de embeddings (PCA 2D)\ncoloreado por cluster DE EMBEDDINGS")
axes[0].legend(fontsize=8, markerscale=2)

for c in range(5):
    m = merged_pca["CLUSTER_ESTRUCTURAL"] == c
    axes[1].scatter(merged_pca.loc[m, "PC1"], merged_pca.loc[m, "PC2"], s=10, alpha=0.6, color=paleta5[c], label=f"C{c}")
axes[1].set_title("Espacio de embeddings (PCA 2D)\ncoloreado por cluster ESTRUCTURAL (05)")
axes[1].legend(fontsize=8, markerscale=2)

plt.tight_layout()
plt.show()


**Lectura:** un ARI cercano a 0 indicaría que los embeddings capturan una estructura
prácticamente independiente de la estructural (alta complementariedad); un ARI cercano a 1
indicaría que agrupan a las personas de forma casi idéntica (redundancia, poco aporte adicional).
El valor obtenido (ver celda anterior) y el gráfico de la derecha — que muestra si los clusters
estructurales forman regiones reconocibles o aparecen mezclados en el espacio semántico — se toman
en conjunto para la conclusión de la sección 9.


## 9. ¿Aportan los embeddings algo que las features estructuradas no capturan?

Además del ARI global, conviene revisar si el espacio semántico distingue **dentro** de un mismo
cluster estructural — por ejemplo, dos docentes de "alta carga docente" (mismo Cluster 3 en `05`)
pueden enseñar/investigar en dominios completamente distintos (ingeniería vs. ciencias sociales), y
esa diferencia de dominio es justamente lo que las 100 columnas de `X_modelado` no capturan (son
conteos de actividad, no de contenido/tema).


In [ ]:
# Dispersión semántica dentro de cada cluster estructural: similitud coseno promedio
# entre pares de personas del mismo cluster vs. pares de clusters distintos.
from itertools import combinations
rng = np.random.default_rng(42)

def similitud_promedio_pares(indices, n_pares=300):
    if len(indices) < 2:
        return np.nan
    pares = rng.choice(len(indices), size=(min(n_pares, len(indices) * (len(indices) - 1) // 2), 2))
    pares = pares[pares[:, 0] != pares[:, 1]]
    sims = [float(X_emb[indices[i]] @ X_emb[indices[j]]) for i, j in pares]
    return np.mean(sims)

idx_por_cluster_estructural = {
    c: [list(ids_con_texto).index(i) for i in comparacion.loc[comparacion["CLUSTER_ESTRUCTURAL"] == c, "IDPERSONA"]]
    for c in sorted(comparacion["CLUSTER_ESTRUCTURAL"].unique())
}

filas_dispersión = []
for c, idxs in idx_por_cluster_estructural.items():
    filas_dispersión.append({
        "CLUSTER_ESTRUCTURAL": c,
        "N_PERSONAS": len(idxs),
        "SIMILITUD_SEMANTICA_INTRA_CLUSTER": similitud_promedio_pares(idxs),
    })

todos_los_idx = list(range(len(ids_con_texto)))
sim_global = similitud_promedio_pares(todos_los_idx, n_pares=500)

dispersión_df = pd.DataFrame(filas_dispersión)
print(f"Similitud semántica promedio entre dos personas cualesquiera (referencia global): {sim_global:.3f}")
dispersión_df


**Interpretación:** si la similitud semántica promedio **dentro** de cada cluster estructural
fuera muy superior a la similitud global de referencia, significaría que el clustering estructural
(basado en volumen/tipo de actividad) ya agrupa implícitamente a personas con temas afines — los
embeddings aportarían poco. Si en cambio la similitud intra-cluster es parecida a la global, cada
cluster estructural mezcla temas muy distintos entre sí, y una representación semántica sí aportaría
una dimensión de información **adicional** (a qué se dedica temáticamente cada persona) que
`X_modelado` no captura.

**Resultado obtenido:** ARI = **0.116** entre la partición estructural y la de embeddings (ambas
K=5) — muy lejos de 1, señal de particiones prácticamente independientes. La similitud semántica
intra-cluster (referencia global: 0.693) es:

- Cluster 1 (perfil administrativo): 0.786 — el más homogéneo temáticamente, esperable dado que el
  vocabulario administrativo/laboral es más acotado que el académico.
- Clusters 0, 2 y 3 (perfiles docentes): 0.72-0.75 — apenas por encima del global, es decir, dentro
  de cada uno de estos clusters conviven temas/disciplinas bastante distintos.
- Cluster 4 (ingreso reciente): 0.578 — el más heterogéneo temáticamente, incluso por debajo del
  promedio global: el criterio que define este cluster (poca antigüedad) no tiene ninguna relación
  con el área de conocimiento de la persona.

**Conclusión de esta exploración (06):**

- Los embeddings **sí capturan información distinta** a la estructural: el ARI de 0.116 confirma
  particiones mayormente independientes, y salvo el caso parcial del cluster administrativo, la
  similitud intra-cluster estructural no supera de forma relevante a la similitud global — es decir,
  los clusters estructurales (definidos por volumen/tipo de actividad: docencia, investigación,
  administración, antigüedad) en general NO predicen bien de qué tema/disciplina trata el trabajo de
  cada persona. Ambas representaciones son **complementarias**, no redundantes: la estructural
  describe *cuánto y qué tipo* de actividad tiene una persona; la semántica describe *sobre qué*
  trata esa actividad.
- **No se concatena una matriz combinada (100 + 384 dimensiones) en este notebook.** Aunque hay
  evidencia de complementariedad, concatenar directamente estructurado + embeddings sin un análisis
  adicional (p. ej. reducir la dimensionalidad de los embeddings para no dominar la distancia
  euclídea frente a las 100 columnas, decidir una ponderación relativa, y re-evaluar
  estabilidad/interpretabilidad del clustering resultante) no está metodológicamente justificado
  todavía — así lo pide la consigna del proyecto. Queda documentado como una línea de trabajo futura
  concreta, no como algo ya resuelto.
- **Uso recomendado en el corto plazo:** mantener ambas representaciones **separadas**. Los perfiles
  estructurales de `06_clustering` siguen siendo la base para el dashboard (`08`); los embeddings de
  este notebook pueden usarse de forma independiente para funciones complementarias como "buscar
  personas con experiencia temática similar a X" (búsqueda semántica) dentro de cada perfil
  estructural, sin fusionar ambas matrices.

## 10. Resumen

**Archivos generados en `data/embeddings/`:**

| Archivo | Contenido |
|---|---|
| `documento_semantico_persona.csv` | `IDPERSONA, DOCUMENTO_TEXTO, SECCIONES_INCLUIDAS, SECCIONES_RECORTADAS, N_SECCIONES, N_CARACTERES, N_PALABRAS` — el documento que se embebe (DEC-014) |
| `corpus_texto_detalle.csv` | `IDPERSONA, FUENTE, TEXTO` — un registro por fragmento, usado por el dashboard como evidencia, no como insumo del embedding |
| `cobertura_texto_personas.csv` | Cobertura de texto por persona (conteos), del corpus por fragmento |
| `embeddings_personas.csv` | `IDPERSONA` + 768 columnas `E_000..E_767` (un embedding por persona, normalizado, `intfloat/multilingual-e5-base`) |
| `embeddings_metadata.csv` | Modelo, dimensiones, método, cobertura |

**Decisiones tomadas (DEC-014, ver `context/DECISION_LOG.md`):**

1. Se reemplazó la agregación de fragmentos (promedio balanceado por fuente) por un
   **documento semántico único por persona**, construido por secciones con información
   real, conservando la relación temporal entre trayectoria dentro/fuera de ESPOL y
   distinguiendo cargo contractual de función adicional/subrogación.
2. Se cambió el modelo de `paraphrase-multilingual-MiniLM-L12-v2` (128 tokens, tuneado
   para parafraseo/STS) a **`intfloat/multilingual-e5-base`** (512 tokens, tuneado para
   recuperación/búsqueda semántica) — decisión técnica: el documento narrativo completo
   no cabía en 128 tokens. Ambos son locales, sin API key (se mantiene DEC-002).
3. Se aplicó un presupuesto de longitud (~380 palabras) que recorta secciones completas de
   menor prioridad (nunca trayectoria ni formación) para las personas con historiales más
   extensos, en vez de dejar que el tokenizador trunque a mitad de frase de forma opaca.
4. Se comparó la estructura de los embeddings contra la jerarquía estructural actual y se
   concluyó que son **complementarios**: no se fusionan en una sola matriz (instrucción
   explícita del usuario de no mezclar clustering con embeddings).

**Limitaciones:**

- Personas sin ninguna sección con información (ver conteo en la sección 5) no tienen
  documento ni embedding — deben tratarse explícitamente en cualquier uso posterior.
- El presupuesto de longitud recorta secciones completas de menor prioridad (nunca
  trayectoria/formación) para el ~20% de la población con historiales más extensos (ver
  sección 5); el recorte aplica tanto al texto guardado en `documento_semantico_persona.csv`
  como al embedding — no se conserva una versión "completa" sin recortar aparte. El detalle
  completo de esas secciones sigue disponible en las fuentes originales
  (`data/processed/*.csv`) y en `corpus_texto_detalle.csv`.
- No se evaluaron alternativas de modelo con contexto aún mayor (p. ej. `BAAI/bge-m3`,
  8192 tokens) ni pooling ponderado por recencia; son posibles mejoras futuras.